# Terminal Case Study — Phase 3: Full Gate + Crane Pipeline

**Case study**: Intermodal Container Terminal | **Phase**: 3 of 5

## Learning Objectives
By the end of this notebook you will be able to:
1. Use the full `TerminalModel` from the `simdes` package.
2. Interpret all four output metrics: `mean_total_time`, `mean_wait_gate`, `mean_wait_crane`, `n_trucks`.
3. Determine which stage (gate vs. crane) is the binding bottleneck.
4. Verify Little's Law for the full system.

---
> Phase 3 adds the crane/yard stage: trucks queue for a crane after clearing the gate.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from simdes.models.terminal import TerminalModel, TerminalParams
from simdes.analysis import confidence_interval

## Full System Description

```
Truck arrival
  │
  ▼
[Queue] ─► 2 Gate agents (Exp 5 min/truck)
  │
  ▼
[Queue] ─► 1 Crane (Exp 8 min/truck)
  │
  ▼
Exit
```

| Stage | Servers | Mean svc | Effective throughput | ρ |
|---|---|---|---|---|
| Gate | 2 | 5 min | 2×(1/5) = 0.400/min | 0.417 |
| Crane | 1 | 8 min | 1/8 = 0.125/min | **0.667** |

Arrival rate: 10 trucks/hour = 1/6 per minute.
The **crane** is the bottleneck.

In [ ]:
baseline = TerminalParams(
    n_gates=2,
    n_cranes=1,
    arrival_rate=10.0,
    gate_mean=5.0,
    crane_mean=8.0,
    sim_time=8.0,
)

model = TerminalModel(params=baseline, seed=42)
df = model.run_replications(30)
df.head()

In [ ]:
# Summary statistics
wait_cols = ['mean_wait_gate', 'mean_wait_crane', 'mean_total_time']
for col in wait_cols:
    m, lo, hi = confidence_interval(df[col].to_numpy())
    print(f'{col:25s}: {m:6.2f} min  95% CI [{lo:.2f}, {hi:.2f}]')

print(f'\nMean trucks served per day: {df["n_trucks"].mean():.1f}')

In [ ]:
# Box plots — per-stage waits
fig, ax = plt.subplots(figsize=(7, 4))
labels = ['Gate\nwait', 'Crane\nwait', 'Total\ntime']
data   = [df[c].to_numpy() for c in wait_cols]
ax.boxplot(data, labels=labels, patch_artist=True,
           boxprops=dict(facecolor='tab:orange', alpha=0.6))
ax.set_ylabel('Mean time per replication (min)')
ax.set_title('Phase 3 — Full terminal: per-stage waits across 30 replications')
ax.grid(axis='y', alpha=0.3)
fig.tight_layout()
plt.show()

In [ ]:
# Little's Law check: L = λW
lam = 10.0 / 60.0   # trucks per minute
W_hat = df['mean_total_time'].mean()
L_littles = lam * W_hat
print(f"Little's Law: L = λW = {lam:.4f} × {W_hat:.2f} = {L_littles:.3f} trucks in system (avg)")

In [ ]:
# Time breakdown: service vs waiting
svc_total  = 5.0 + 8.0   # expected service times
wait_total = df['mean_wait_gate'].mean() + df['mean_wait_crane'].mean()
total      = df['mean_total_time'].mean()

fig, ax = plt.subplots(figsize=(4, 5))
categories = ['Gate svc', 'Gate wait', 'Crane svc', 'Crane wait']
values     = [5.0, df['mean_wait_gate'].mean(), 8.0, df['mean_wait_crane'].mean()]
colors     = ['#1f77b4', '#aec7e8', '#ff7f0e', '#ffbb78']
bottom = 0
for v, c, l in zip(values, colors, categories):
    ax.bar('Truck journey', v, bottom=bottom, color=c, label=f'{l}: {v:.1f} min')
    bottom += v
ax.set_ylabel('Minutes')
ax.set_title('Average truck time breakdown')
ax.legend(loc='upper left', fontsize=8, bbox_to_anchor=(1.01, 1))
fig.tight_layout()
plt.show()

## Summary

The crane is the dominant bottleneck:
- The gate wait is minimal (2 gates handle λ = 10 trucks/hr with ρ = 0.42).
- The crane queue accumulates because ρ_crane = 0.667 — moderate but visible.
- Most of a truck's time in the terminal is waiting for or receiving crane service.

In Phase 4 we will run scenarios to find the optimal gate/crane combination.

## Try It Yourself

1. Change `n_cranes=2`. How much does the crane wait drop?
2. At what arrival rate does the crane utilisation reach 0.9?
3. Verify Little's Law separately for each stage. Do both hold?